In [4]:
import pandas as pd
from datetime import datetime

In [5]:
df = pd.read_csv("../data/raw/quikr_car.csv")
df.head()

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel


In [10]:
df.shape

(892, 6)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        892 non-null    object
 1   company     892 non-null    object
 2   year        892 non-null    object
 3   Price       892 non-null    object
 4   kms_driven  840 non-null    object
 5   fuel_type   837 non-null    object
dtypes: object(6)
memory usage: 41.9+ KB


In [13]:
df.describe()

,name,company,year,Price,kms_driven,fuel_type
count,892,892,892,892,840,837
unique,525,48,61,274,258,3
top,Honda City,Maruti,2015,Ask For Price,"45,000 kms",Petrol
freq,13,235,117,35,30,440


In [14]:
df.isna().sum()

name           0
company        0
year           0
Price          0
kms_driven    52
fuel_type     55
dtype: int64

In [20]:
df["Price"].value_counts().head(10)

Price
Ask For Price    35
2,50,000         17
3,50,000         14
1,80,000         13
1,30,000         12
4,00,000         12
3,00,000         11
4,50,000         11
1,50,000         11
2,20,000         11
Name: count, dtype: int64

In [19]:
df["kms_driven"].value_counts().head(10)

kms_driven
45,000 kms    30
35,000 kms    30
55,000 kms    25
50,000 kms    23
20,000 kms    22
60,000 kms    20
30,000 kms    19
40,000 kms    18
65,000 kms    16
38,000 kms    16
Name: count, dtype: int64

In [21]:
df["fuel_type"].value_counts(dropna=False)

fuel_type
Petrol    440
Diesel    395
NaN        55
LPG         2
Name: count, dtype: int64

In [22]:
# чистим цены
def clean_price(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value.lower() == "ask for price":
        return None

    value = value.replace(",", "")

    try:
        return int(value)
    except ValueError:
        return None

# чистим пробег
def clean_kms(value):
    if pd.isna(value):
        return None

    value = str(value).lower().strip()
    value = value.replace("kms", "")
    value = value.replace("km", "")
    value = value.replace(",", "")
    value = value.strip()

    try:
        return int(value)
    except ValueError:
        return None

In [27]:
df["price"] = df["Price"].apply(clean_price)
df["kms_driven_num"] = df["kms_driven"].apply(clean_kms)
df["year_num"] = pd.to_numeric(df["year"], errors="coerce")
current_year = datetime.now().year
df["car_age"] = current_year - df["year_num"]
df["fuel_type"] = df["fuel_type"].fillna("Unknown")
df["has_price"] = df["price"].notna()

In [29]:
def get_price_segment(price):
    if pd.isna(price):
        return "No price"

    if price < 100_000:
        return "0-100k"
    if price < 300_000:
        return "100k-300k"
    if price < 500_000:
        return "300k-500k"
    if price < 800_000:
        return "500k-800k"

    return "800k+"

def get_kms_segment(kms):
    if pd.isna(kms):
        return "Unknown"

    if kms < 25_000:
        return "0-25k"
    if kms < 50_000:
        return "25k-50k"
    if kms < 100_000:
        return "50k-100k"

    return "100k+"

In [30]:
df["price_segment"] = df["price"].apply(get_price_segment)
df["kms_segment"] = df["kms_driven_num"].apply(get_kms_segment)

In [31]:
result_columns = [
    "name",
    "company",
    "year_num",
    "car_age",
    "price",
    "kms_driven_num",
    "fuel_type",
    "price_segment",
    "kms_segment",
    "has_price",
]

In [32]:
df_clean = df[result_columns]
df_clean.head()

,name,company,year_num,car_age,price,kms_driven_num,fuel_type,price_segment,kms_segment,has_price
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007.0,19.0,80000.0,45000.0,Petrol,0-100k,25k-50k,True
1,Mahindra Jeep CL550 MDI,Mahindra,2006.0,20.0,425000.0,40.0,Diesel,300k-500k,0-25k,True
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018.0,8.0,NaN,22000.0,Petrol,No price,0-25k,False
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014.0,12.0,325000.0,28000.0,Petrol,300k-500k,25k-50k,True
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014.0,12.0,575000.0,36000.0,Diesel,500k-800k,25k-50k,True


In [37]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   name            892 non-null    object
 1   company         892 non-null    object
 2   year_num        842 non-null    Int64 
 3   car_age         842 non-null    Int64 
 4   price           857 non-null    Int64 
 5   kms_driven_num  838 non-null    Int64 
 6   fuel_type       892 non-null    object
 7   price_segment   892 non-null    object
 8   kms_segment     892 non-null    object
 9   has_price       892 non-null    bool  
dtypes: Int64(4), bool(1), object(5)
memory usage: 67.2+ KB


In [41]:
df_clean = df[result_columns].copy()
df_clean["year_num"] = df_clean["year_num"].astype("Int64")
df_clean["car_age"] = df_clean["car_age"].astype("Int64")
df_clean["kms_driven_num"] = df_clean["kms_driven_num"].astype("Int64")
df_clean["price"] = df_clean["price"].astype("Int64")

In [50]:
df_clean.head()

,name,company,year_num,car_age,price,kms_driven_num,fuel_type,price_segment,kms_segment,has_price
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,19,80000,45000,Petrol,0-100k,25k-50k,True
1,Mahindra Jeep CL550 MDI,Mahindra,2006,20,425000,40,Diesel,300k-500k,0-25k,True
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,8,<NA>,22000,Petrol,No price,0-25k,False
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,12,325000,28000,Petrol,300k-500k,25k-50k,True
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,12,575000,36000,Diesel,500k-800k,25k-50k,True


In [54]:
df["company"].count()

['2012',
 '7',
 '9',
 'Any',
 'Audi',
 'BMW',
 'Chevrolet',
 'Commercial',
 'Datsun',
 'Fiat',
 'Force',
 'Ford',
 'Hindustan',
 'Honda',
 'Hyundai',
 'I',
 'Jaguar',
 'Jeep',
 'Land',
 'MARUTI',
 'Mahindra',
 'Maruti',
 'Mercedes',
 'Mini',
 'Mitsubishi',
 'Nissan',
 'Renault',
 'Sale',
 'Skoda',
 'Swift',
 'TATA',
 'Tara',
 'Tata',
 'Toyota',
 'URJENT',
 'Used',
 'Volkswagen',
 'Volvo',
 'Well',
 'Yamaha',
 'all',
 'i',
 'scratch',
 'sell',
 'selling',
 'tata',
 'urgent',
 'very']

In [48]:
df_clean["company"].value_counts()

valid_companies = [
    "Maruti",
    "Hyundai",
    "Mahindra",
    "Tata",
    "Honda",
    "Toyota",
    "Chevrolet",
    "Renault",
    "Ford",
    "Volkswagen",
    "Skoda",
    "Audi",
    "Mini",
    "Datsun",
    "BMW",
    "Mitsubishi",
    "Mercedes",
    "Nissan",
    "Force",
    "Fiat",
    "Hindustan",
    "Jaguar",
    "Land",
    "Jeep",
    "Volvo",
]

df_clean = df_clean[df_clean["company"].isin(valid_companies)].copy()

In [49]:
df_clean.to_csv("../data/clean/used_cars_clean.csv", index=False)